# 02 Plate Solving

**Eingabe**: kalibriertes Referenzbild (erster Frame der Nacht, bestes Seeing)  
**Ausgabe**: WCS (RA/Dec ↔ Pixel) als ASTAP-Datei `<bild>.wcs` neben dem Bild — das Originalbild bleibt unverändert

Werkzeug: **ASTAP-CLI** (`astap -f <datei> -r 10 -wcs`)  
Binary auf macOS: `/Applications/ASTAP.app/Contents/MacOS/astap`  
Download + Datenbank (D05/D80): https://www.hnsky.org/astap.htm

> **Echtes Bild**: `VARPHOT_REF=/pfad/bild.fit` setzen (oder `REF_OVERRIDE` in Schritt 1) → ASTAP löst es.  
> **Demo-Modus**: Die synthetischen Demo-FITS haben WCS bereits im Header.  
> ASTAP findet kein echtes Sternmuster → Schritt 3 fällt automatisch auf den Header-WCS zurück.

## Schritt 1 — Pfade laden

Liest `session_config.json` aus Notebook 01, ermittelt den Pfad zu den kalibrierten Frames
und wählt das erste Frame als Referenzbild für das Plate Solving.

In [ ]:
import os, json
from pathlib import Path
from astropy.io import fits
from astropy.wcs import WCS
from varphot import solve_image
from varphot.astrometry import ASTAP_DEFAULT

# ASTAP-Binary auf macOS (liegt im .app-Bundle)
ASTAP = ASTAP_DEFAULT

# Optional: echtes Einzelbild direkt angeben (überschreibt den Siril-Workspace)
REF_OVERRIDE = os.environ.get("VARPHOT_REF")   # z. B. "demo/real_m16/m16_ha_benchmark.fit"

if REF_OVERRIDE:
    REF = Path(REF_OVERRIDE).expanduser().resolve()
    if not REF.exists():
        raise FileNotFoundError(f"REF_OVERRIDE nicht gefunden: {REF}")
    WORKSPACE, DIR_CAL, cal_frames = REF.parent, REF.parent, [REF]
else:
    # session_config.json aus Notebook 01 laden
    _cfgs = list(Path.home().rglob("session_config.json"))
    if not _cfgs:
        raise FileNotFoundError("session_config.json nicht gefunden — bitte Notebook 01 ausführen.")
    session_cfg = json.loads(_cfgs[0].read_text())
    WORKSPACE = Path(session_cfg["workspace"])
    DIR_CAL   = Path(session_cfg["dir_cal"])

    _FITS_GLOB = ["*.fits", "*.fit", "*.fts", "*.FITS", "*.FIT", "*.FTS"]
    cal_frames = sorted(f for g in _FITS_GLOB for f in DIR_CAL.glob(g))
    if not cal_frames:
        raise FileNotFoundError(f"Keine FITS in {DIR_CAL} — Siril-Kalibrierung abgeschlossen?")
    REF = cal_frames[0]

print(f"ASTAP:              {ASTAP}  ({'OK' if Path(ASTAP).exists() else 'NICHT GEFUNDEN'})")
print(f"Workspace:          {WORKSPACE}")
print(f"Kalibrierte Bilder: {len(cal_frames)} in {DIR_CAL}")
print(f"Referenzbild:       {REF}")

## Schritt 2 — Referenzbild anzeigen

Zeigt das kalibrierte Referenzbild mit logarithmischer Streckung (Percentile 1–99.5 %).
So werden sowohl schwache Hintergrundsterne als auch helle Objekte sichtbar.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.visualization import ZScaleInterval

# Große Bilder (z. B. 9252×6946 px) nur für die Anzeige jeden n-ten Pixel — die Daten bleiben unverändert
def load_preview(path, max_px=2500):
    with fits.open(path) as hdul:
        data = hdul[0].data
        hdr = hdul[0].header.copy()
        ny, nx = data.shape[-2:]
        f = max(1, int(np.ceil(max(nx, ny) / max_px)))
        small = np.asarray(data[..., ::f, ::f], dtype=float)
    return small, hdr, nx, ny, f

img_small, hdr, NX, NY, STRIDE = load_preview(REF)
vmin, vmax = ZScaleInterval().get_limits(img_small)

fig, ax = plt.subplots(figsize=(12, 8))
ax.imshow(img_small, origin="lower", cmap="gray", vmin=vmin, vmax=vmax, interpolation="nearest",
          extent=(-0.5, NX - 0.5, -0.5, NY - 0.5))
ax.set_title(f"Referenzbild: {REF.name}  |  {NX}×{NY} px  |  "
             f"EXPTIME={hdr.get('EXPTIME','?')}s  FILTER={hdr.get('FILTER','?')}"
             + (f"  |  Anzeige 1:{STRIDE}" if STRIDE > 1 else ""), fontsize=11)
ax.set_xlabel("x [px]")
ax.set_ylabel("y [px]")
plt.tight_layout()
plt.show()
print(f"ZScale: {vmin:.4g} – {vmax:.4g} (Einheiten des Bildes)")


## Schritt 3 — Plate Solving mit ASTAP

ASTAP vergleicht die Sternpositionen im Bild mit einer lokalen Sternkatalog-Datenbank
(D05 für schwache Felder, D80 für helle Felder wie Delta Cep) und schreibt den
gefundenen WCS in eine Datei `<bild>.wcs` neben dem Bild (Funktion `varphot.solve_image`). Das Originalbild wird nicht verändert.

**Demo-Modus**: Synthetische Bilder haben kein echtes Sternmuster → ASTAP schlägt fehl.
Die Zelle fällt in diesem Fall automatisch auf den vorhandenen Header-WCS zurück.

In [ ]:
solve = solve_image(REF, astap=ASTAP, radius_deg=10)
print(solve.log if solve.log else "(keine ASTAP-Ausgabe)")
for n in solve.notes:
    print("⚠ ", n)

if solve.wcs is None:
    raise RuntimeError("Weder ASTAP noch FITS-Header liefern eine WCS — Bild nicht auswertbar (Sterne? Datenbank? -r?)")

WCS_SOURCE = {"ASTAP": "ASTAP", "header": "header (ASTAP fehlgeschlagen)"}.get(solve.source, solve.source)
print(f"\nWCS-Quelle: {WCS_SOURCE}")


## Schritt 4 — WCS prüfen und visualisieren

Zeigt eine **HTML-Tabelle** mit den wichtigsten WCS-Kennzahlen, danach das Referenzbild
mit **RA/Dec-Koordinatengitter** (cyan). Zwei Markierungen:

- **Grau `+`** = Sensorzentrum (geometrische Mitte des Chips)
- **Gelber Kreis `○`** = Zielstern-Position aus FITS-Header (`OBJECT` + `CRVAL`) — ungefähre Lage, exakte Koordinaten folgen in Notebook 03 via SIMBAD

In [ ]:
import astropy.units as u
import numpy as np
from astropy.coordinates import SkyCoord
from astropy.visualization import ZScaleInterval
from IPython.display import display, HTML

w = solve.wcs                      # WCS aus Schritt 3 (nicht aus dem Originalheader)
img_data_wcs, hdr, nx, ny, _ = load_preview(REF)
hdr_img = fits.getheader(REF)       # Aufnahme-Header (OBJECT, RA/DEC der Montierung)

# --- WCS-Kennzahlen ---
cx, cy = nx / 2, ny / 2
center_ra, center_dec = w.all_pix2world([[cx, cy]], 0)[0]
ps  = w.proj_plane_pixel_scales()
scale_as = float(ps[0].to(u.arcsec).value)
fov_ra   = nx * scale_as / 60
fov_dec  = ny * scale_as / 60
pc       = w.wcs.get_pc()
rot_deg  = np.degrees(np.arctan2(pc[0, 1], pc[0, 0]))
center_coord = SkyCoord(ra=center_ra*u.deg, dec=center_dec*u.deg)
center_hms   = center_coord.to_string("hmsdms", sep=":")

obj_name = hdr_img.get("OBJECT", "unbekannt")
# Zielposition: Montierungs-Zeigeposition RA/DEC aus dem Aufnahme-Header (Grad); sonst Bildmitte.
# (CRVAL ist der WCS-Referenzpunkt, nicht das Ziel.) Exakte Koordinaten folgen in NB03 (SIMBAD).
if "RA" in hdr_img and "DEC" in hdr_img:
    tgt_ra, tgt_dec, TGT_SRC = float(hdr_img["RA"]), float(hdr_img["DEC"]), "Header RA/DEC"
else:
    tgt_ra, tgt_dec, TGT_SRC = center_ra, center_dec, "Bildmitte (kein RA/DEC im Header)"
tgt_px, tgt_py = w.all_world2pix([[tgt_ra, tgt_dec]], 0)[0]

# --- HTML-Tabelle ---
src_color = "#4caf50" if WCS_SOURCE == "ASTAP" else "#ff9800"
display(HTML(f"""
<table style="border-collapse:collapse;font-family:monospace;font-size:13px;margin:8px 0">
  <tr style="background:#1e1e2e;color:#cdd6f4">
    <th colspan="2" style="padding:6px 16px;text-align:center;letter-spacing:1px">
      WCS — Plate-Solve-Ergebnis</th></tr>
  <tr><td style="padding:4px 12px;color:#888">Quelle</td>
      <td style="padding:4px 12px;color:{src_color}"><b>{WCS_SOURCE}</b></td></tr>
  <tr style="background:#11111b"><td style="padding:4px 12px;color:#888">Sensorzentrum</td>
      <td style="padding:4px 12px">RA {center_ra:.4f}°  Dec {center_dec:+.4f}°
        <span style="color:#888">({center_hms})</span></td></tr>
  <tr><td style="padding:4px 12px;color:#888">Zielstern</td>
      <td style="padding:4px 12px"><b>{obj_name}</b>
        — Position: {TGT_SRC}; exakte Koordinaten folgen in NB03 (SIMBAD)</td></tr>
  <tr style="background:#11111b"><td style="padding:4px 12px;color:#888">Pixelskala</td>
      <td style="padding:4px 12px">{scale_as:.3f} arcsec/px</td></tr>
  <tr><td style="padding:4px 12px;color:#888">Bildfeld</td>
      <td style="padding:4px 12px">{fov_ra:.1f}′ × {fov_dec:.1f}′</td></tr>
  <tr style="background:#11111b"><td style="padding:4px 12px;color:#888">Rotation</td>
      <td style="padding:4px 12px">{rot_deg:.2f}°</td></tr>
  <tr><td style="padding:4px 12px;color:#888">Bildgröße</td>
      <td style="padding:4px 12px">{nx} × {ny} px</td></tr>
</table>"""))

# --- WCS-Plot mit Koordinatengitter ---
interval = ZScaleInterval()
vmin, vmax = interval.get_limits(img_data_wcs)

fig = plt.figure(figsize=(12, 8.5))
# Platz oben für RA-Achse (WCSAxes), unten für Dec-Achse → kein ax.set_title()
ax = fig.add_subplot(1, 1, 1, projection=w)
fig.subplots_adjust(top=0.88)   # Freiraum oben für RA-Label

ax.imshow(img_data_wcs, origin="lower", cmap="gray", extent=(-0.5, nx - 0.5, -0.5, ny - 0.5),
          vmin=vmin, vmax=vmax, interpolation="nearest", aspect="equal")

# RA/Dec-Gitternetz
overlay = ax.get_coords_overlay("icrs")
overlay.grid(color="cyan", linestyle="--", linewidth=0.6, alpha=0.7)
overlay[0].set_axislabel("RA (J2000)", fontsize=10, minpad=0.5)
overlay[1].set_axislabel("Dec (J2000)", fontsize=10)
overlay[0].set_ticks(number=6)
overlay[1].set_ticks(number=5)
overlay[0].set_ticklabel(size=9)
overlay[1].set_ticklabel(size=9)

tr = ax.get_transform("pixel")

# Sensorzentrum: graues Kreuz
ax.plot(cx, cy, "+", color="#aaaaaa", markersize=12, markeredgewidth=1.2,
        transform=tr, label="Sensorzentrum")

# Zielstern: gelber Kreis + Beschriftung
ax.plot(tgt_px, tgt_py, "o", color="yellow", markersize=22,
        markeredgewidth=1.5, markerfacecolor="none",
        transform=tr, label=f"Zielstern: {obj_name}")
ax.annotate(obj_name, xy=(tgt_px, tgt_py),
            xytext=(tgt_px + 35, tgt_py + 35),
            xycoords=tr, textcoords=tr,
            color="yellow", fontsize=9, fontweight="bold",
            arrowprops=dict(arrowstyle="-", color="yellow", lw=0.8))

# Titel als Text-Box INNERHALB des Bildes (oben links) → kein Konflikt mit WCS-Achsen
info = (f"{obj_name}  |  {REF.name}\n"
        f"{fov_ra:.1f}′ × {fov_dec:.1f}′  |  {scale_as:.2f}″/px  |  WCS: {WCS_SOURCE}")
ax.text(0.01, 0.98, info, transform=ax.transAxes,
        fontsize=8.5, color="white", va="top", ha="left",
        bbox=dict(boxstyle="round,pad=0.3", facecolor="black", alpha=0.55))

# Achsenbeschriftung: WCSAxes ordnet RA/Dec je nach Bilddrehung der x- bzw. y-Achse zu
for _c in ax.coords:
    _c.set_axislabel("RA (J2000)" if _c.coord_type == "longitude" else "Dec (J2000)", fontsize=10)
ax.legend(loc="lower right", fontsize=9, framealpha=0.7)
plt.show()

## Schritt 5 — Sternbild-Kontext

Das Bildfeld (typisch 20′–60′) ist für sichtbare Sternbildlinien zu klein. Diese Zelle zeigt
deshalb eine **generische Übersichtskarte ~25° × 25°** um den Zielstern:

- Alle 88 IAU-Sternbilder mit Standard-Verbindungslinien (Quelle: `data/constellation_lines.json`)
- Beim ersten Aufruf werden die Sternpositionen via SIMBAD abgefragt und gecacht (~1–3 Min)
- Folgeaufrufe lesen nur den Cache — kein Netz nötig
- Ein Sternbild wird gezeichnet, sobald eine seiner Linien das Kartenfenster berührt; Linien werden als Großkreisbögen projiziert und am Fensterrand abgeschnitten (nicht verworfen), auch wenn beide Endsterne außerhalb liegen
- Rotes FOV-Rechteck zeigt den tatsächlichen Bildausschnitt

In [ ]:
import json, subprocess, sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.lines as mlines

# ---------------------------------------------------------------------------
# constellation_lines.json laden; bei erstem Aufruf auto-generieren
# ---------------------------------------------------------------------------
_ROOT   = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
_LINES  = _ROOT / "data" / "constellation_lines.json"
_SCRIPT = _ROOT / "tools" / "make_const_data.py"

if not _LINES.exists():
    print("⏳ Erst-Setup: Sternpositionen via SIMBAD abfragen (~1–3 Min) …")
    result = subprocess.run(
        [sys.executable, str(_SCRIPT)],
        capture_output=False, cwd=str(_ROOT)
    )
    if result.returncode != 0 or not _LINES.exists():
        raise RuntimeError(
            f"make_const_data.py fehlgeschlagen. "
            f"Manuell ausführen: python {_SCRIPT.relative_to(_ROOT)}"
        )

const_data = json.loads(_LINES.read_text())

# ---------------------------------------------------------------------------
# Zielkoordinaten (aus Schritt 4)
# ---------------------------------------------------------------------------
RA0  = tgt_ra  if "tgt_ra"  in dir() and tgt_ra  else center_ra
DEC0 = tgt_dec if "tgt_dec" in dir() and tgt_dec else center_dec
HALF = 13.5    # halbe Kartenbreite in Grad (±13.5° → ~27° Ausschnitt)

# ---------------------------------------------------------------------------
# Gnomische Projektion (korrekt für Himmelskoordinaten)
# ---------------------------------------------------------------------------
from varphot import gnomonic as gnom, great_circle_xy, polyline_in_window   # NaN für die abgewandte Halbkugel

# ---------------------------------------------------------------------------
# Sternbilder auswählen: ein Sternbild zählt, sobald eine seiner Linien das Kartenfenster berührt
# (auch wenn beide Endsterne außerhalb liegen — Linien werden am Rand abgeschnitten, nicht verworfen)
# ---------------------------------------------------------------------------

# Feste Farbpalette: rotiert durch alle sichtbaren Sternbilder
_PALETTE = [
    "#64b5f6","#ffcc80","#ce93d8","#80cbc4","#ef9a9a",
    "#a5d6a7","#fff176","#80deea","#f48fb1","#b39ddb",
]

# ---------------------------------------------------------------------------
# Plot aufbauen
# ---------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(11, 11), facecolor="#07070f")
ax.set_facecolor("#07070f")
ax.set_aspect("equal")
ax.set_xlim(-HALF, HALF)
ax.set_ylim(-HALF, HALF)

legend_items = []
color_idx    = 0
drawn        = 0

for abbrev, entry in const_data.items():
    stars  = entry["stars"]
    lines  = entry["lines"]
    cname  = entry["name"]
    if not stars:
        continue

    # Linien als Großkreisbögen projizieren (Geraden in der Gnomonik); hinten liegende Teile sind NaN
    segs = []
    for s1, s2 in lines:
        if s1 not in stars or s2 not in stars:
            continue
        gx, gy = great_circle_xy(stars[s1]["ra"], stars[s1]["dec"], stars[s2]["ra"], stars[s2]["dec"],
                                 RA0, DEC0, n=48)
        if polyline_in_window(gx, gy, HALF):
            segs.append((gx, gy))
    if not segs:
        continue

    col = _PALETTE[color_idx % len(_PALETTE)]
    color_idx += 1
    drawn     += 1

    # Linien zeichnen — der Achsenbereich schneidet sie am Kartenrand ab
    for gx, gy in segs:
        ax.plot(gx, gy, "-", color=col, linewidth=1.1, alpha=0.65, zorder=2, solid_capstyle="round")

    # Sterne zeichnen
    label_placed = False
    lx_sum, ly_sum, ln = 0.0, 0.0, 0
    for sname, sd in stars.items():
        x, y = gnom(sd["ra"], sd["dec"], RA0, DEC0)
        if not (np.isfinite(x) and np.isfinite(y)):
            continue
        if abs(x) > HALF * 1.5 or abs(y) > HALF * 1.5:
            continue
        mag  = sd.get("mag", 5.0)
        size = max(8, (5.5 - min(mag, 5.0)) ** 2 * 10)
        # Zielstern hervorheben (Stern am nächsten zu RA0/DEC0 im Bereich < 2°)
        dist2 = (x ** 2 + y ** 2)
        is_tgt = dist2 < 4.0    # 2°-Radius
        ax.scatter(x, y, s=size,
                   color="yellow" if is_tgt else "white",
                   edgecolors="yellow" if is_tgt else "none",
                   linewidths=1.0, zorder=4 if is_tgt else 3)
        lx_sum += x; ly_sum += y; ln += 1

    # Sternbild-Name: Zentrum der sichtbaren Sterne, sonst der im Fenster sichtbaren Linienpunkte
    if not ln:
        px = np.concatenate([g[0] for g in segs]); py = np.concatenate([g[1] for g in segs])
        vis = np.isfinite(px) & np.isfinite(py) & (np.abs(px) < HALF * 0.95) & (np.abs(py) < HALF * 0.95)
        if vis.any():
            lx_sum, ly_sum, ln = px[vis].sum(), py[vis].sum(), int(vis.sum())
    if ln:
        lx, ly = lx_sum / ln, ly_sum / ln
        if abs(lx) < HALF * 0.95 and abs(ly) < HALF * 0.95:
            ax.text(lx, ly, cname, fontsize=9, color=col, alpha=0.5,
                    ha="center", va="center", fontweight="bold", style="italic", zorder=1)

    legend_items.append(mlines.Line2D([], [], color=col, lw=1.5, label=cname))

# ---------------------------------------------------------------------------
# FOV-Rechteck
# ---------------------------------------------------------------------------
fov_ra_deg  = fov_ra  / 60    # Bogenminuten → Grad
fov_dec_deg = fov_dec / 60
ax.add_patch(mpatches.Rectangle(
    (-fov_ra_deg / 2, -fov_dec_deg / 2), fov_ra_deg, fov_dec_deg,
    linewidth=1.8, edgecolor="#ff6666", facecolor="none", zorder=6
))
ax.text(0, -fov_dec_deg / 2 - 0.4,
        f"FOV {fov_ra:.1f}′×{fov_dec:.1f}′", color="#ff6666",
        fontsize=8, ha="center", va="top", zorder=6)

# Zielmarkierung im Zentrum
ax.scatter(0, 0, s=120, color="yellow", edgecolors="yellow",
           linewidths=1.5, marker="o", facecolors="none", zorder=7)
ax.scatter(0, 0, s=20,  color="yellow", zorder=7)

# ---------------------------------------------------------------------------
# Gitternetz, Achsen, Legende, Titel
# ---------------------------------------------------------------------------
for d in np.arange(-12, 13, 3):
    ax.axhline(d, color="#ffffff", linewidth=0.2, alpha=0.12, zorder=1)
    ax.axvline(d, color="#ffffff", linewidth=0.2, alpha=0.12, zorder=1)

legend_items.append(mpatches.Patch(edgecolor="#ff6666", facecolor="none",
                                    linewidth=1.5, label="Bildfeld"))
ax.legend(handles=legend_items, loc="lower right", fontsize=8,
          framealpha=0.35, labelcolor="white", facecolor="#111")

obj_name = hdr.get("OBJECT", "Ziel")
ax.set_xlabel("RA-Offset vom Zielstern [°]  (links = Ost)", color="#aaa", fontsize=10)
ax.set_ylabel("Dec-Offset vom Zielstern [°]", color="#aaa", fontsize=10)
ax.tick_params(colors="#555")
for sp in ax.spines.values():
    sp.set_edgecolor("#333")

ax.set_title(
    f"Sternbild-Kontext: {obj_name}  |  "
    f"RA {RA0:.3f}°  Dec {DEC0:+.3f}°  |  "
    f"±{HALF}°  ({drawn} Sternbilder)",
    color="white", fontsize=11, pad=10
)
plt.tight_layout()
plt.show()
print(f"Quelle: {_LINES.name}  |  {drawn} Sternbilder dargestellt")

## Schritt 6 — Sterne erkennen und Sternkarte

`varphot.detect()` findet die Sterne im Bild (Hintergrund ortsabhängig, `background="2d"`,
damit Nebelstrukturen nicht als Sterne gelten; Schwelle `NSIGMA` gegen das **lokale** Rauschen).
Die Quellenliste wird als `<bild>.sources.ecsv` neben dem Bild gespeichert und nur neu berechnet,
wenn das Bild neuer ist (`skip_if_fresh`) — Eingabe für Notebook 03.

`plot_detections()` zeigt links das ganze Bild mit den hellsten Quellen, rechts einen Ausschnitt
1:1 (Kreise = erkannte Sterne, rote Nummern = die 10 hellsten) mit RA/Dec-Gitter der WCS aus Schritt 3.
Passen die Kreise auf die Sterne, stimmen Erkennung **und** WCS.

Parameter: `FWHM_PX` = Stern-FWHM in Pixeln (für M16-Testbild gemessen: ≈ 5 px, mit `photutils.psf.fit_fwhm`);
`NSIGMA` = 25 für Nebelfelder, 8 für leere Sternfelder.

In [ ]:
import numpy as np
from astropy.table import Table
from varphot import detect, plot_detections, skip_if_fresh

FWHM_PX, NSIGMA = 5.0, 25          # Stern-FWHM [px]; Schwelle in Sigma des lokalen Rauschens

SOURCES = REF.with_suffix(".sources.ecsv")
img_full = fits.getdata(REF)

if skip_if_fresh(SOURCES, REF):
    stars = Table.read(SOURCES)
    print(f"Quellenliste aktuell → geladen: {SOURCES.name} ({len(stars)} Quellen)")
else:
    stars, sky_med, sky_std = detect(img_full, fwhm=FWHM_PX, nsigma=NSIGMA, background="2d")
    if stars is None:
        raise RuntimeError("Keine Sterne erkannt — FWHM_PX/NSIGMA prüfen")
    stars.write(SOURCES, overwrite=True)
    print(f"{len(stars)} Quellen erkannt (Himmel {sky_med:.4g}, Rauschen {sky_std:.3g}) → {SOURCES.name}")

fig = plot_detections(img_full, solve.wcs, stars, fwhm=FWHM_PX, target=(tgt_ra, tgt_dec))
plt.show()

# die 10 hellsten Quellen mit Himmelskoordinaten (zur Gegenprüfung z. B. in Stellarium/Aladin)
top = stars[np.argsort(stars["flux"])[::-1][:10]]
sky = solve.wcs.pixel_to_world(top["x_centroid"], top["y_centroid"])
for k, (x, y, fl, c) in enumerate(zip(top["x_centroid"], top["y_centroid"], top["flux"], sky), 1):
    print(f"{k:2d}  x={x:7.1f} y={y:7.1f}  RA {c.ra.to_string('hour', precision=1)}  "
          f"Dec {c.dec.to_string(precision=0, alwayssign=True)}  Fluss {fl:.3g}")

## Schritt 7 — WCS gegen Gaia prüfen

ASTAP meldet nur sein eigenes Residuum. Hier wird die WCS **unabhängig** geprüft: Die erkannten Sterne aus Schritt 6 werden mit Gaia-DR3-Sternen
(G 9–14, RUWE < 1,4, Eigenbewegung auf die Aufnahmeepoche gerechnet) gekreuzt; gegenseitig nächste Nachbarn innerhalb 2,5″ zählen als Treffer.

Gezeigt werden: Versatz (Median), Streuung (robust), Wiederfindungsrate und die **schlechteste Bildzelle** (Gitter 8×6; erkennt Rotations-/Skalenfehler,
die der Gesamtversatz verdeckt). Urteil in Pixeln der Bildskala: Versatz ≤ 0,5 px, Streuung ≤ 0,7 px, schlechteste Zelle ≤ 1 px (Startwerte).
Braucht Netz (Gaia-TAP: ESA, bei Ausfall ARI-Spiegel); das Ergebnis wird in `~/.cache/varphot` zwischengespeichert.
Lerneffekt am M16-Testbild: mit ASTAP-Standard (500 Sterne) blieb ein Rotationsfehler von ~1,2″ am Bildrand, `solve_image()` nutzt deshalb jetzt 5000 Sterne.

In [ ]:
from astropy.time import Time
from varphot import check_wcs, gaia_field, plot_wcs_check

if WCS_SOURCE != "ASTAP":
    print("WCS stammt nicht aus einer ASTAP-Lösung (Demo/Header) — Gaia-Prüfung nur mit echtem Bild sinnvoll.")
else:
    # Aufnahmeepoche: Mitte von DATE-OBS und (falls vorhanden) DATE-END
    t0 = Time(hdr_img["DATE-OBS"])
    t1 = Time(hdr_img["DATE-END"]) if "DATE-END" in hdr_img else t0
    EPOCH = (t0.jyear + t1.jyear) / 2

    # Gaia-Kegel um die Bildmitte, groß genug für die Bilddiagonale
    scale_deg = scale_as / 3600
    radius = 0.5 * float(np.hypot(NX, NY)) * scale_deg + 0.05
    gaia = gaia_field(center_ra, center_dec, radius, gmax=14.0, gmin=9.0)

    chk = check_wcs(solve.wcs, (stars["x_centroid"], stars["y_centroid"]), gaia, EPOCH, (NY, NX))
    print(f"Epoche {EPOCH:.2f} | {len(gaia)} Gaia-Sterne im Kegel, {chk.n_gaia_in_image} im Bild, {chk.n_match} Treffer "
          f"({100 * chk.recovery:.0f} % wiedergefunden)")
    print(f"Versatz (erkannt − Gaia): RA·cosδ {chk.median_dra_arcsec:+.2f}″  Dec {chk.median_ddec_arcsec:+.2f}″  "
          f"| Streuung {chk.rms_arcsec:.2f}″ ({chk.rms_arcsec / chk.pixel_scale_arcsec:.2f} px)")
    print(f"schlechteste Bildzelle {chk.worst_cell_arcsec:.2f}″ | linearer Trend über das Bild {chk.trend_arcsec:.2f}″")
    print("Urteil:", chk.verdict)
    plot_wcs_check(chk, (NY, NX))
    plt.show()